# CD8 T Cell Factor Interpretation: SemanticSCVI

Gaublomme-style factor interpretation for PixelGen CD8 T cells using SemanticSCVI
with functional protein embeddings.

| Cell | Description |
|------|-------------|
| 1 | Setup, load data, validate embeddings, train SemanticSCVI, health check |
| 2 | Pseudotime & UMAP |
| 3 | GEP discovery (Leiden + characterization + plots) |
| 4 | Factor diagnostics: orthogonality, marker heatmap, marker modules |
| **5** | **Select modules & score (edit `SELECTED_MODULES` here)** |
| 6 | Compute associations & correlations |
| 7 | Correlation heatmap |
| 8 | Factor scatter grid (edit `f1`, `f2`, `COLOR_ITEMS`) |
| 9 | Quadrant subpopulation analysis (edit `QF1`, `QF2`) |

In [ ]:
# Cell 1 — Setup, Load Data, Validate Embeddings, Train/Load Model, Health Check
import os, sys
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[k] = '1'

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from scvi.model._semantic_scvi import SemanticSCVI

# --- Paths ---
_PROJECT_DIR = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
_FACTOR_DIR  = _PROJECT_DIR / 'factor_analysis'
sys.path.insert(0, str(_FACTOR_DIR))
sys.path.insert(0, str(_PROJECT_DIR))

import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import (
    CACHE_DIR, CD8_ADATA_PATH,
    configure_mpl, plot_embedding_distances,
    filter_non_t_markers, prepare_adata_for_model, extract_model_outputs, plot_training_health,
)

configure_mpl()

# ======================== CONFIGURATION ========================
SEED = 42
scvi.settings.seed = SEED
RESULTS_DIR = _FACTOR_DIR / 'results_semantic'
RESULTS_DIR.mkdir(exist_ok=True)
SEMANTIC_MODEL_DIR = CACHE_DIR / 'semantic_scvi_cd8_model'

N_TOP   = 80
N_LATENT = 10
FORCE_RETRAIN = True            # >>> EDIT: set True to retrain from scratch <<<

# --- Model architecture ---
N_HIDDEN               = 128
GENE_LIKELIHOOD        = 'nb'
COHERENCE_WEIGHT       = 2000.0
LOSS_MODE              = 'geometric'
N_GENE_SAMPLE          = None    # None → min(1024, n_vars)
USE_DECODER_BATCH_NORM = False
DECORRELATION_LOSS_WEIGHT = 120.0  # W-loading orthogonality penalty; default 120, raise (e.g. 500-2000) to reduce Z factor correlation
Z_DECOR_WEIGHT            = 50.0    # direct Z-covariance off-diag penalty; 0 = off (safe default); try 10-100 if Z factors still correlated

# --- Training ---
MAX_EPOCHS              = 500
WARMUP_EPOCHS           = 30
WARMUP_SCHEDULE         = 'cosine'
BATCH_SIZE              = 128
TRAIN_SIZE              = 0.9
EARLY_STOPPING          = True
EARLY_STOPPING_PATIENCE = 30
CHECK_VAL_EVERY_N_EPOCH = 5

# --- Non-T-cell marker filter (applied BEFORE top-N variable selection) ---
REMOVE_NON_T_MARKERS = True   # >>> EDIT: drop canonical non-T lineage markers before HVG selection <<<
NON_T_CELL_MARKERS = {
    'B_cell':           ['CD19','CD20','CD21','CD22','CD72','CD180','CD79a','IgM','IgD','CD268','CD10','CD24'],
    'Plasma_cell':      ['CD138','CD269','CD319'],
    'Monocyte_macro':   ['CD14','CD64','CD163','CD206','CD13','CD33',],
    'DC':               ['CD1a','CD1b','CD1c','CD141','CD209','CD371','CD169'],
    'Granulocyte':      ['CD193','CD89','CD66b'],
    'NK_restricted':    ['NKp80','CD335','CD337',],
    'Mast_basophil':    ['CD117','IgE'],
    'Platelet_endoth':  ['CD41','CD62P','CD31'],
    'Stromal_epithel':  ['CD326',],
    'Progenitor':       ['CD34'],
    'APC_costim':       ['CD40','CD80','CD86','CD35','CD37','CD123'],
}
# ===============================================================

print(f'scvi-tools: {scvi.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')

# --- Load data ---
adata = sc.read_h5ad(CD8_ADATA_PATH)
print(f'\nLoaded {adata.n_obs:,} cells x {adata.n_vars} markers')
print(f'Layers: {list(adata.layers.keys())}')
print(f'Obs columns: {list(adata.obs.columns)}')

emb_path = _FACTOR_DIR / 'functional_protein_embeddings.pkl'
with open(emb_path, 'rb') as f:
    emb_data = pickle.load(f)
print(f'Embeddings: {emb_data["embeddings"].shape} ({len(emb_data["dimensions"])} dims)')

plot_embedding_distances(emb_data, results_dir=RESULTS_DIR)

# --- Filter canonical non-T-cell markers ---
if REMOVE_NON_T_MARKERS:
    adata = filter_non_t_markers(adata, NON_T_CELL_MARKERS)

# --- Prepare adata & semantic map ---
adata, semantic_map = prepare_adata_for_model(adata, emb_data, n_top=N_TOP)

# --- Train or load SemanticSCVI ---
SemanticSCVI.setup_anndata(adata, layer='counts', batch_key=None)

n_gene_sample = N_GENE_SAMPLE or min(1024, adata.n_vars)
model_kwargs = dict(
    semantic_map=semantic_map.float(),
    n_latent=N_LATENT,
    n_hidden=N_HIDDEN,
    gene_likelihood=GENE_LIKELIHOOD,
    coherence_weight=COHERENCE_WEIGHT,
    loss_mode=LOSS_MODE,
    n_gene_sample=n_gene_sample,
    use_decoder_batch_norm=USE_DECODER_BATCH_NORM,
    decorrelation_loss_weight=DECORRELATION_LOSS_WEIGHT,
    z_decor_weight=Z_DECOR_WEIGHT,
)

if SEMANTIC_MODEL_DIR.exists() and not FORCE_RETRAIN:
    print(f'Loading cached SemanticSCVI from {SEMANTIC_MODEL_DIR}')
    model = SemanticSCVI(adata, **model_kwargs)
    model_state = torch.load(SEMANTIC_MODEL_DIR / 'model.pt', map_location='cpu', weights_only=False)
    state_dict = {k: v for k, v in model_state['model_state_dict'].items()
                  if k in set(model.module.state_dict().keys())}
    model.module.load_state_dict(state_dict)
    model.is_trained_ = True
    print('Loaded cached model.')
else:
    print('Training SemanticSCVI from scratch...')
    model = SemanticSCVI(adata, **model_kwargs)
    print(model)
    model.train(
        max_epochs=MAX_EPOCHS,
        warmup_epochs=WARMUP_EPOCHS,
        warmup_schedule=WARMUP_SCHEDULE,
        batch_size=BATCH_SIZE,
        train_size=TRAIN_SIZE,
        early_stopping=EARLY_STOPPING,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    )
    SEMANTIC_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save(SEMANTIC_MODEL_DIR, overwrite=True)
    print(f'Model saved to {SEMANTIC_MODEL_DIR}')

# --- Extract latents & weights ---
Z_arr, W_arr, W_df, marker_names, z_var = extract_model_outputs(model, adata, N_LATENT)
n_factors = N_LATENT

plot_training_health(model, results_dir=RESULTS_DIR)
print(f'\nResults dir: {RESULTS_DIR}')


In [ ]:
# Cell 2 — Pseudotime & UMAP
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import compute_pseudotime, plot_umap_overview

pseudotime = compute_pseudotime(adata, Z_arr, seed=SEED)
plot_umap_overview(adata, meta_cols=['dpt_pseudotime','condition','time','cell_system'], seed=SEED, results_dir=RESULTS_DIR)

In [ ]:
sc.pl.umap(adata, layer='arcsinh',color=['CD3e','CD8','TCRVd2','CD4'],)

In [ ]:
# Cell 3 — GEP Discovery (Leiden clustering + characterization + plots)
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import run_gep_discovery

N_TOP = 100
programs, program_char_df = run_gep_discovery(
    W_arr, marker_names,
    n_top=N_TOP, n_neighbors=10, resolution=2.5,
    seed=SEED, results_dir=RESULTS_DIR,
)

In [ ]:
# Cell 4 — Factor diagnostics: orthogonality + marker heatmap + marker modules
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_factor_diagnostics

N_TOP_MARKER = 50
N_TOP_MODULE = 100
MAX_K = 15

module_info = plot_factor_diagnostics(
    Z_arr, W_arr, marker_names,
    n_factors=n_factors,
    n_top_marker=N_TOP_MARKER,
    n_top_module=N_TOP_MODULE,
    max_k=MAX_K,
    results_dir=RESULTS_DIR,
)

---

## USER CHECKPOINT

**Review the GEPs above.** Set `SELECTED_MODULES` in the next cell, then run Phase 2.

---

In [ ]:
# Cell 5 — Select modules & compute scores (hierarchical modules + canonical)
# >>> EDIT SELECTED_MODULES after reviewing Cell 4 module output <<<
SELECTED_MODULES = list(module_info['module_genes'].keys())  # default: all

import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import compute_program_scores, score_programs, MARKER_PROGRAMS

prog_scores_df = compute_program_scores(
    adata, module_info['module_genes'],
    selected_programs=SELECTED_MODULES,
    layer='arcsinh',
)
# Rename columns from GEP_<int> to Module_<int> for clarity
prog_scores_df.columns = [c.replace('GEP_', 'Module_') for c in prog_scores_df.columns]
print(f'Module scores: {prog_scores_df.shape}')
for col in prog_scores_df.columns:
    mid = int(col.split('_')[1])
    markers = module_info['module_genes'][mid]
    print(f'  {col} ({len(markers)} markers): {", ".join(markers[:8])}{"..." if len(markers) > 8 else ""}')

print(f'\nScoring {len(MARKER_PROGRAMS)} canonical programs...')
canonical_scores_df = score_programs(adata, MARKER_PROGRAMS, layer='arcsinh')
print(f'Canonical program scores: {canonical_scores_df.shape}')

In [ ]:
# Cell 6 — Compute all associations & correlations
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import (compute_metadata_associations, factor_program_correlation,
                          build_interpretation_table)

assoc_df = compute_metadata_associations(Z_arr, adata, z_var)
corr_df, pval_df, padj_df = factor_program_correlation(Z_arr, prog_scores_df)
canon_corr_df, canon_pval_df, canon_padj_df = factor_program_correlation(Z_arr, canonical_scores_df)
interp_df = build_interpretation_table(assoc_df, corr_df, padj_df, program_char_df=None)
print(f'Computed: {n_factors} factors x {len(assoc_df.columns)} metadata tests, '
      f'{corr_df.shape[1]} modules, {canon_corr_df.shape[1]} canonical GEPs.')
display(interp_df)

In [ ]:
# Cell 7 — Correlation heatmap (metadata + data GEPs + canonical GEPs)
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_correlation_heatmap

plot_correlation_heatmap(
    assoc_df, corr_df, padj_df,
    canon_corr_df=canon_corr_df,
    canon_padj_df=canon_padj_df,
    results_dir=RESULTS_DIR,
)

In [ ]:
# Cell 8 — Factor scatter grid (edit f1/f2 for axes, COLOR_ITEMS for panels)
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_factor_scatter_grid

# --- Pick factor axes ---
top2 = np.argsort(z_var)[::-1][:2]
f1, f2 = int(top2[0]), int(top2[1])

# >>> EDIT f1, f2 to override <<<
f1 = 7
f2 = 9


# --- Color transform for continuous variables ---
# "winsorize" (default): clip 2nd-98th percentile, rescale
# "quantile": map to percentile rank
# "none": raw values
COLOR_TRANSFORM = "winsorize"

# --- Pick what to color by ---
COLOR_ITEMS = [
    # Metadata
    'meta:condition',
    'meta:time',
    'meta:cell_system',
    'meta:dpt_pseudotime',
    # Data-derived GEPs
    *[f'gep:{c}' for c in prog_scores_df.columns],
    # Canonical programs
    *[f'canonical:{c}' for c in canonical_scores_df.columns],
]

plot_factor_scatter_grid(
    Z_arr, adata,
    color_items=COLOR_ITEMS,
    prog_scores_df=prog_scores_df,
    canonical_scores_df=canonical_scores_df,
    f1=f1, f2=f2,
    max_pts=2000, seed=SEED,
    color_transform=COLOR_TRANSFORM,
    results_dir=RESULTS_DIR,
    save_name='factor_scatter_grid.png',
)

In [ ]:
# Cell 9 — Quadrant subpopulation analysis
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import run_quadrant_analysis

# --- Pick factor axes and the variable whose gradient defines the quadrants ---
QF1, QF2 = f1, f2                    # >>> EDIT <<<
# Available canonical programs:
print(f'Canonical programs: {list(canonical_scores_df.columns)}')
QUAD_VARIABLE = canonical_scores_df.columns[3]  # >>> EDIT: pick from list above <<<
print(f'Using: {QUAD_VARIABLE}')
color_vals = canonical_scores_df[QUAD_VARIABLE].values

quadrant_results = run_quadrant_analysis(
    Z_arr, adata, f1=QF1, f2=QF2,
    prog_scores_df=prog_scores_df,
    canonical_scores_df=canonical_scores_df,
    color_vals=color_vals,
    color_name=QUAD_VARIABLE,
    method='regression',
    n_top_de=20,
    results_dir=RESULTS_DIR,
)


These markers form the pSMAC (peripheral Supramolecular Activation Cluster), providing the structural "Velcro" required to restrain the target cell during the lethal strike.
* **`CD11a` & `CD18` (LFA-1):** The primary master integrin complex for stable T-cell adhesion.
* **`CD29` & `CD49D` (VLA-4):** The secondary crucial integrin pair that locks the immunological synapse.
* **`CD50` (ICAM-3):** A key adhesion molecule that sustains the physical membrane seal.

This triad represents the transition from early activation into a terminal, highly lethal effector state.
* **`GPR56`** *(Highest Fold-Change)*: A definitive marker for highly cytotoxic, antigen-experienced lymphocytes. It restricts proliferation but maximizes direct killing capacity.
* **`CX3CR1`**: Defines a highly specific subset of effector memory CD8+ T cells that contain the maximum levels of intracellular granzyme and perforin.
* **`CD226` (DNAM-1):** A dominant activating receptor that drives the physical killing of tumor cells by binding to stress ligands (e.g., CD155) on the leukemia blasts.

* **`CD154` (CD40L):** While canonically a CD4+ marker, its expression on CD8+ T cells indicates extreme, acute activation. The T cell uses this to engage CD40 on the B-ALL blasts, forcing the target into an inflammatory, vulnerable state.
